# Check the DRP stage1 (single-visit) output collection

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-11 : stage 1 
- purpose : find the user collection written by `DRP.yaml#stage1-single-visit`, list what it contains, and look at a few products (images, star catalogs)

## Contents

1. Run description
2. Setup
3. Collections
4. Content of the stage1 collection
5. Pipeline steps (quantum diagrams)
6. Explore the products : `post_isr_image`, `single_visit_star`, `single_visit_star_detector`, image + catalog
7. Next notebooks

## 1. Run description

- `file bps_dp2cosmos.sh`

```bash
export VISITIDS=$(paste -sd, cosmos_small.txt)
export CCDIDS=$(paste -sd, ccdIds.txt)

release="w_2026_43"

echo "run BPS"
bps_submit.sh --interactive bps_dp2cosmos_parsl.yaml
```

- `fichier bps_dp2cosmos_parsl.yaml`

```
pipelineYaml: "${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit"

payload:
  runInit: true
  payloadName: dp2cosmos202611
  butlerConfig: main
  inCollection: LSSTCam/defaults
  output: u/${USER}/LSSTCam/runelaiss1_20261111_tract2877patchsel_RunCosmos9patches/stage1
  dataQuery: "instrument='LSSTCam' AND skymap='lsst_cells_v2' AND tract IN (${TRACTIDS})  AND patch IN (${PATCHIDS}) AND detector IN (${CCDIDS}) AND exposure.observation_type='science'"
```

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## 2. Setup

In [4]:
import os
import logging
import warnings
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from lsst.daf.butler import Butler

import showplots_utils
importlib.reload(showplots_utils)   # pick up any edit of the helper module without restarting the kernel
from showplots_utils import show_diagram

In [5]:
# Turn the Butler.collections FutureWarning into an error
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

# Logging of the Rubin Science Pipelines (INFO is verbose : use logging.WARNING for a quiet notebook)
LOG_LEVEL = logging.INFO
logging.basicConfig(
    level=LOG_LEVEL,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)
logging.getLogger('lsst').setLevel(LOG_LEVEL)

butler = Butler('main')

###### u/dagoret/LSSTCam/runelaiss1_20261111_tract2877patchsel_RunCosmos9patches/stage1

In [6]:
username = os.getenv('USER')
RUN_NAME = 'runelaiss1_20261111_tract2877patchsel_RunCosmos9patches'

collection_path = f'u/{username}/LSSTCam/{RUN_NAME}/stage1'

NPRINT = 10              # number of dataIds printed per dataset type
COUNT_DATASETS = True    # count the datasets of each type (can take a while on a big collection)

# Quantum diagrams of the pipeline steps
DIAGRAM_DIR = Path('../../../../notebooks/fig_diagrams')

# Selection for the image + catalog example of section 6.4 (None = first one available)
VISIT_ID = None
DETECTOR_ID = None

print('user       :', username)
print('collection :', collection_path)

user       : dagoret
collection : u/dagoret/LSSTCam/runelaiss1_20261111_tract2877patchsel_RunCosmos9patches/stage1


In [7]:
def query_refs(dataset_type, collection=None):
    '''All the dataset references of a dataset type in the collection.'''
    return list(
        butler.registry.queryDatasets(
            datasetType=dataset_type,
            collections=collection or collection_path,
            findFirst=False,
        )
    )


def print_refs(refs, n=NPRINT):
    '''Number of references and the first n dataIds.'''
    print(f'{len(refs)} dataset(s)')
    for ref in refs[:n]:
        print('  ', ref.dataId)
    if len(refs) > n:
        print(f'   ... ({len(refs) - n} more)')


def dval(ref, key):
    '''Value of a dimension in the dataId of a reference (None if absent).'''
    try:
        return ref.dataId[key]
    except KeyError:
        return None


def to_pandas(table):
    '''Catalog (astropy / arrow table) to DataFrame.'''
    if isinstance(table, pd.DataFrame):
        return table
    return table.to_pandas()

## 3. Collections

In [8]:
# Existing user collections
user_collections = sorted(butler.collections.query(f'u/{username}/*'))

for name in user_collections:
    print(name)

print(f'Number of collections : {len(user_collections)}')

u/dagoret/LSSTCam/runcosmos01_20261014_data202505_RunCosmosDemo/stage1
u/dagoret/LSSTCam/runcosmos01_20261014_data202505_RunCosmosDemo/stage1/20261014T133218Z
u/dagoret/LSSTCam/runcosmos01_20261014_data202505_RunCosmosDemo/stage2
u/dagoret/LSSTCam/runcosmos01_20261014_data202505_RunCosmosDemo/stage2/20261016T200927Z
u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1
u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1/20261027T202157Z
u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage2
u/dagoret/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage2/20261030T144305Z
u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/fgcm_nb01/20261111T123747Z
u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1
u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1/20261105T102248Z
u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_

In [9]:
# Check the chosen collection (stops here if it does not exist)
if collection_path not in butler.collections.query(collection_path):
    raise RuntimeError(f'Collection does not exist : {collection_path}')

info = butler.collections.get_info(collection_path)
print(f'Collection found : {collection_path}')
print(f'Type             : {info.type}')

MissingCollectionError: No collection with name 'u/dagoret/LSSTCam/runelaiss1_20261111_tract2877patchsel_RunCosmos9patches/stage1' found.

## 4. Content of the stage1 collection

One row per dataset type : number of datasets, dimensions and storage class. The dataset types are taken from the collection summary, then counted type by type (no need to load the references of the ~44 million datasets in memory). If the summary is not available, all the datasets are scanned.

In [ ]:
def dataset_types_in_collection(collection):
    '''Dataset type names of the collection summary (None if the summary is not available).'''
    try:
        info = butler.collections.get_info(collection, include_summary=True)
        return sorted(info.dataset_types)
    except Exception as exc:
        print('Collection summary not available :', exc)
        return None


def count_refs(dataset_type):
    '''Number of datasets of a type in the collection.'''
    try:
        return butler.registry.queryDatasets(
            datasetType=dataset_type, collections=collection_path, findFirst=False
        ).count()
    except Exception:
        return len(query_refs(dataset_type))


type_names = dataset_types_in_collection(collection_path)

rows = []
if type_names is None:
    from collections import Counter
    counter = Counter(r.datasetType.name for r in query_refs('*'))
    for name, n in sorted(counter.items()):
        rows.append(dict(datasetType=name, n=n, dimensions='', storageClass=''))
else:
    for name in type_names:
        dt = butler.get_dataset_type(name)
        rows.append(dict(
            datasetType=name,
            n=count_refs(name) if COUNT_DATASETS else None,
            dimensions=', '.join(dt.dimensions.names),
            storageClass=dt.storageClass_name,
        ))

df_types = pd.DataFrame(rows).set_index('datasetType')

print(f'Collection : {collection_path}')
print(f'Number of dataset types : {len(df_types)}')
print(f'  of which plots (Plot suffix) : {int(df_types.index.str.endswith("Plot").sum())}')
if COUNT_DATASETS:
    print(f'Total number of datasets : {int(df_types["n"].sum())}')

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 250):
    display(df_types)

In [ ]:
# Filter the table by a string in the dataset type name (e.g. 'star', 'visit_image', 'Plot', 'metric')
NAME_FILTER = 'star'

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 250):
    display(df_types[df_types.index.str.contains(NAME_FILTER, case=False)])

## 5. Pipeline steps (quantum diagrams)

In [ ]:
DIAGRAM_STEPS = {
    'step1a-single-visit-detectors': 'DRP step1a/detectors',
    'step1b-single-visit-visits':    'DRP step1b/visits',
    'step1c-single-visit-tracts':    'DRP step1c/tracts',
    'step1d-single-visit-global':    'DRP step1d/global',
}

for step, label in DIAGRAM_STEPS.items():
    show_diagram(DIAGRAM_DIR / f'pipeline_drp_pipe_LSSTCam_DRP_{step}.png', title=label)

## 6. Explore the products

Star-related dataset types of the stage1 collection (descriptions from the pipeline documentation, to be checked against the quantum diagrams above) :

| Dataset type | Content / role |
|---|---|
| `single_visit_star` | consolidated catalog of the stars of a visit, product of the star association |
| `single_visit_star_detector` | star catalog associated with the individual detectors |
| `isolated_star` | samples of isolated stars, used in particular for photometric and astrometric calibration |
| `isolated_star_association` | result of the association of isolated stars across observations |
| `single_visit_psf_star` | stars selected for the PSF modeling (a specific selection, not a general stellar catalog) |

Suggested order : `single_visit_star`, then `single_visit_star_detector` to study the stars detected on each CCD.

### 6.1 `post_isr_image`

In [ ]:
refs_isr = query_refs('post_isr_image')
print_refs(refs_isr)

### 6.2 `single_visit_star` catalog

In [ ]:
refs_star = query_refs('single_visit_star')
print_refs(refs_star)

if refs_star:
    catalog_star = butler.get(refs_star[0])
    print('type   :', type(catalog_star))
    print('length :', len(catalog_star))

    df_star = to_pandas(catalog_star)
    display(df_star.head())
    print(df_star.columns.tolist())

### 6.3 `single_visit_star_detector` : stars detected per detector

In [ ]:
refs_star_det = query_refs('single_visit_star_detector')
print_refs(refs_star_det)

if refs_star_det:
    catalog_star_det = butler.get(refs_star_det[0])
    print('type   :', type(catalog_star_det))
    print('length :', len(catalog_star_det))

    df_star_det = to_pandas(catalog_star_det)
    display(df_star_det.head())
    print(df_star_det.columns.tolist())

### 6.4 Image and catalog of one detector

Find the (visit, detector) pairs present in both datasets. The `post_isr_image` dataId is labelled by `exposure` and the star catalog by `visit` ; for LSSTCam the two identifiers are equal.

In [ ]:
refs_img = refs_isr
refs_cat = refs_star_det

print('image dimensions   :', list(refs_img[0].dataId.dimensions.names))
print('catalog dimensions :', list(refs_cat[0].dataId.dimensions.names))

img_by_key = {(dval(r, 'exposure'), dval(r, 'detector')): r for r in refs_img}
cat_by_key = {(dval(r, 'visit'), dval(r, 'detector')): r for r in refs_cat}

common_keys = sorted(img_by_key.keys() & cat_by_key.keys())

print(f'images : {len(img_by_key)}, catalogs : {len(cat_by_key)}, common (visit, detector) pairs : {len(common_keys)}')
print('visits    :', sorted({k[0] for k in common_keys}))
print('detectors :', sorted({k[1] for k in common_keys}))

In [ ]:
# Selection : VISIT_ID / DETECTOR_ID of the configuration cell (None = first available)
candidates = [
    k for k in common_keys
    if (VISIT_ID is None or k[0] == VISIT_ID) and (DETECTOR_ID is None or k[1] == DETECTOR_ID)
]
if not candidates:
    raise RuntimeError(f'No common (visit, detector) pair for VISIT_ID={VISIT_ID}, DETECTOR_ID={DETECTOR_ID}')

key = candidates[0]
ref_img, ref_cat = img_by_key[key], cat_by_key[key]

exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)
df_cat = to_pandas(catalog)

print('visit, detector  :', key)
print('image   dataId   :', ref_img.dataId)
print('catalog dataId   :', ref_cat.dataId)
print('image type       :', type(exposure))
print('image shape      :', exposure.image.array.shape)
print('catalog type     :', type(catalog))
print('catalog length   :', len(df_cat))
print('catalog columns  :', df_cat.columns.tolist())

Display of the image, with the catalog stars overlaid when pixel centroid columns are found. The overlay assumes the catalog centroids are in the pixel frame of the image : check that the circles sit on sources, otherwise read the column list above and adapt `find_xy_columns`.

In [ ]:
def find_xy_columns(df):
    '''Names of the pixel centroid columns of a catalog, or None.'''
    names = {c.lower(): c for c in df.columns}
    for cx, cy in [('x', 'y'),
                   ('centroid_x', 'centroid_y'),
                   ('slot_centroid_x', 'slot_centroid_y'),
                   ('base_sdsscentroid_x', 'base_sdsscentroid_y'),
                   ('xcentroid', 'ycentroid')]:
        if cx in names and cy in names:
            return names[cx], names[cy]
    return None


img = exposure.image.array
bbox = exposure.getBBox()
extent = (bbox.getMinX() - 0.5, bbox.getMaxX() + 0.5, bbox.getMinY() - 0.5, bbox.getMaxY() + 0.5)
vmin, vmax = np.nanpercentile(img, [5, 99])

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(img, origin='lower', cmap='gray', vmin=vmin, vmax=vmax, extent=extent)
ax.set_xlabel('x (pixel)')
ax.set_ylabel('y (pixel)')

cols = find_xy_columns(df_cat)
if cols is not None:
    ax.scatter(df_cat[cols[0]], df_cat[cols[1]], s=40, facecolors='none', edgecolors='r', linewidths=0.6)
    ax.set_title(f'visit {key[0]}, detector {key[1]} : {len(df_cat)} catalog stars ({cols[0]}, {cols[1]})')
else:
    ax.set_title(f'visit {key[0]}, detector {key[1]} : no x, y centroid columns found in the catalog')
    print('No x, y centroid columns found : see the list of catalog columns above')

plt.show()

## 7. Next notebooks

- `nb2026-11-06_ShowPlots_DRPstage1-single-visit.ipynb` : the `lsst.analysis.tools` plots of this collection
- `nb2026-11-05_CheckCollections_DRPstage2-recalibrate.ipynb` : same check for stage 2
- `nb2026-11-05_PlotPostISRCCD_DRPstage1-single-visit.ipynb` : the post-ISR CCD images